In [1]:
import time
import datetime as dt
from bs4 import BeautifulSoup as bs
import requests

In [2]:
#Function to get all necessary data from game
    #Adapted from pdpharr process_link(url) method (also returns starting hitter stats)
def get_game_data(extension):
    # #Spoof IP in header
    # import random
    # ip = str(random.randint(1, 256)) + '.' + str(random.randint(1, 256)) + '.' + str(random.randint(1, 256)) + '.' + str(random.randint(1, 256)) + '.'
    # headers={'X-Forwarded-For': ip}

    #TEST get data from a single games
    url = 'https://www.baseball-reference.com' + extension
    response = requests.get(url)
    # response = send_request_through_tor(url)
    game_id = url.split('/')[-1][:-6]
    game_id
    soup = bs(response.text)

    #Game Summary
    game_summary = {'game_id': game_id}
    scorebox = soup.find('div', {'class': 'scorebox'})
    #Break loop if page is blank
    if scorebox is None:
        return {}
    strongs = scorebox.find_all('strong')
    game_summary['away_team_abbr'] = strongs[0].find('a')[
                                                     'href'].split('/')[-2]
    game_summary['home_team_abbr'] = strongs[1].find('a')[
                                                     'href'].split('/')[-2]
    meta = scorebox.find('div', {'class': 'scorebox_meta'}).find_all('div')
    #TODO: Add additional summary info for total / score forecasting
    game_summary['date'] = meta[0].text.strip()
    game_summary['start_time'] = meta[1].text[12:-6].strip()

    #Table dict
    #Note: need to preprocess because tables appear in comments in the HTML
    #Note: fuck baseball reference for making this so unnecessarily difficult
    uncommented_html = ''
    for h in response.text.split('\n'):
        # if '<!--     <div' in h: h.replace('<!--     <div', '')
        # if h.strip() == '<!--': h.replace('<!--', '')
        # if h.strip() == '-->': h.replace('-->', '')
        if '<!--     <div' in h:
            continue
        if h.strip() == '<!--':
            continue
        if h.strip() == '-->':
            continue
        uncommented_html += h + '\n'
    soup = bs(uncommented_html)
    stats_tables = soup.find_all('table', {'class': 'stats_table'})

    #Away Batting Table (Table 1)
    a_foot = stats_tables[1].find('tfoot')
    away_team_batting_stats = {x['data-stat']
        : x.text.strip() for x in a_foot.findAll('td')}

    #Home Batting Table (Table 2)
    h_foot = stats_tables[2].find('tfoot')
    home_team_batting_stats = {x['data-stat']
        : x.text.strip() for x in h_foot.findAll('td')}

    #Away / Home Team Pitching Tables (Table 3/4)
    ap_foot = stats_tables[3].find('tfoot')
    away_team_pitching_stats = {
        x['data-stat']: x.text.strip() for x in ap_foot.findAll('td')}
    hp_foot = stats_tables[4].find('tfoot')
    home_team_pitching_stats = {
        x['data-stat']: x.text.strip() for x in hp_foot.findAll('td')}

    #Away Individual Pitcher Table
    ap_table = stats_tables[3]
    away_pitcher_stats = []
    ap_rows = ap_table.find_all('tr')[1:-1]
    for r in ap_rows:
        summary = {x['data-stat']: x.text.strip() for x in r.find_all('td')}
        summary['name'] = r.find(
            'th', {'data-stat': 'player'}).find('a')['href'].split('/')[-1][:-6].strip()
        away_pitcher_stats.append(summary)

    #Home Individual Pitcher Table
    hp_table = stats_tables[4]
    home_pitcher_stats = []
    hp_rows = hp_table.find_all('tr')[1:-1]
    for r in hp_rows:
        summary = {x['data-stat']: x.text.strip() for x in r.find_all('td')}
        summary['name'] = r.find(
            'th', {'data-stat': 'player'}).find('a')['href'].split('/')[-1][:-6].strip()
        home_pitcher_stats.append(summary)

    #Away Individual Hitter Table
    ab_table = stats_tables[1]
    away_hitter_stats = []
    ab_rows = ab_table.find_all('tr')[1:-1]
    for r in ab_rows:
        #Only add starting lineup
        if '\xa0\xa0\xa0' in r.find('th').text:
            continue
        summary = {x['data-stat']: x.text.strip() for x in r.find_all('td')}

        #If non-hitting pitchers in box score
        if r.find('th', {'data-stat': 'player'}).find('a') is None:
            continue

        summary['name'] = r.find(
            'th', {'data-stat': 'player'}).find('a')['href'].split('/')[-1][:-6].strip()
        away_hitter_stats.append(summary)

    #Home Individual Hitter Table
    hb_table = stats_tables[2]
    home_hitter_stats = []
    hb_rows = hb_table.find_all('tr')[1:-1]
    for r in hb_rows:
        #Only add starting lineup
        if '\xa0\xa0\xa0' in r.find('th').text:
            continue
        summary = {x['data-stat']: x.text.strip() for x in r.find_all('td')}

        #If non-hitting pitchers in box score
        if r.find('th', {'data-stat': 'player'}).find('a') is None:
            continue

        summary['name'] = r.find(
            'th', {'data-stat': 'player'}).find('a')['href'].split('/')[-1][:-6].strip()
        home_hitter_stats.append(summary)

    data = {
        'game': game_summary,
        'away_batting': away_team_batting_stats,
        'home_batting': home_team_batting_stats,
        'away_pitching': away_team_pitching_stats,
        'home_pitching': home_team_pitching_stats,
        'away_pitchers': away_pitcher_stats,
        'home_pitchers': home_pitcher_stats,
        #Delta from rdpharr process_link(url) return value below
        'away_hitters': away_hitter_stats,
        'home_hitters': home_hitter_stats
    }
    return data

In [5]:
## Get game links
game_links = []
# Set year to pull game from
current_year = 2016

url = f"https://www.baseball-reference.com/leagues/MLB/{current_year}-schedule.shtml"
resp = requests.get(url)
soup = bs(resp.text)
games = soup.findAll('a', text='Boxscore')
game_links.extend([x['href'] for x in games])
print("Number of games to download: ", len(game_links))

Number of games to download:  2463


In [6]:
# Check specific links
url = 'https://www.baseball-reference.com' + game_links[2402]
# get_game_data(game_links[638])
url



'https://www.baseball-reference.com/boxes/CHA/CHA201610010.shtml'

In [7]:
# Pull data
game_data_2016 = []

for i in range(len(game_data_2016), len(game_links)):
    link = game_links[i]
    # print(link)
    time.sleep(3.2)
    game_data_2016.append(get_game_data(link))
    if len(game_data_2016) % 50 == 0:
        print(dt.datetime.now().time(), len(game_data_2016))

12:16:12.990539 50
12:19:21.539443 100
12:22:25.633819 150
12:25:38.407347 200
12:28:45.907909 250
12:31:48.888971 300
12:34:58.408295 350
12:38:05.812276 400
12:41:12.948629 450
12:44:23.704675 500
12:47:25.732437 550
12:50:31.604442 600
12:53:37.742084 650
12:56:42.437111 700
12:59:49.370295 750
13:02:57.070511 800
13:06:00.360533 850
13:09:04.261056 900
13:12:10.615526 950
13:15:20.193842 1000
13:18:24.702763 1050


In [45]:
# game_data_2015[2403]['home_hitters']
len(game_data_2016[2402]['home_hitters'])

9

In [27]:
### DO NOT RUN / running will OVERRIDE game_data_2015.pkl
# import pickle
# ###pickle.dump(game_data_2015, open('game_data_2015.pkl', 'wb'))

In [ ]:
### DO NOT RUN / running will OVERRIDE game_data_2016.pkl
# import pickle
# ###pickle.dump(game_data_2016, open('game_data_2016.pkl', 'wb'))

In [41]:
gd16 = pickle.load(open('game_data_2016.pkl', 'rb'))

In [43]:
gd16[2402]['home_hitters']


[{'AB': '3',
  'R': '1',
  'H': '1',
  'RBI': '0',
  'BB': '1',
  'SO': '2',
  'PA': '4',
  'batting_avg': '.256',
  'onbase_perc': '.323',
  'slugging_perc': '.398',
  'onbase_plus_slugging': '.721',
  'pitches': '16',
  'strikes_total': '11',
  'wpa_bat': '0.085',
  'leverage_index_avg': '0.96',
  'wpa_bat_pos': '0.123',
  'wpa_bat_neg': '-0.038%',
  'cwpa_bat': '0.02%',
  'cli_avg': '0.45',
  're24_bat': '0.3',
  'PO': '0',
  'A': '0',
  'details': '2B',
  'name': 'hicksaa01'},
 {'AB': '4',
  'R': '0',
  'H': '1',
  'RBI': '1',
  'BB': '0',
  'SO': '1',
  'PA': '4',
  'batting_avg': '.236',
  'onbase_perc': '.307',
  'slugging_perc': '.444',
  'onbase_plus_slugging': '.751',
  'pitches': '13',
  'strikes_total': '9',
  'wpa_bat': '0.125',
  'leverage_index_avg': '1.39',
  'wpa_bat_pos': '0.173',
  'wpa_bat_neg': '-0.048%',
  'cwpa_bat': '0.03%',
  'cli_avg': '0.65',
  're24_bat': '0.4',
  'PO': '2',
  'A': '4',
  'details': 'SB',
  'name': 'doziebr01'},
 {'AB': '4',
  'R': '0',
  'H